<a href="https://colab.research.google.com/github/Mwende-Fifi/nyc-airbnb-market-analysis/blob/main/nyc_airbnb_market_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls /content/drive/MyDrive/portfolio/nyc-airbnb-market-analysis/data/

New_York_AirBnB_listings.csv


# NYC Airbnb Market Analysis (2024)

## 1. Business Context

New York City is one of Airbnb's largest and most regulated markets, with over 20,000 active listings spread across five boroughs and dozens of neighbourhoods. Pricing in this market is not uniform: a studio in Manhattan and a two-bedroom in Staten Island can differ by hundreds of dollars per night, and the drivers of that gap are not obvious from the listing page alone.

For example, two listings can offer similar-sized spaces but command very different prices simply because they are in different neighbourhoods or have different room types. A host needs to know whether those differences are driven mainly by location, property type, or review history before deciding how to price a property. The same patterns can help marketplace teams understand where listings are concentrated and which parts of the city have limited supply.

This analysis asks: what explains price differences across NYC Airbnb listings, and how is supply distributed across the city?


## 2. Data Preparation

### 2.1 Load the data

The dataset is the Inside Airbnb NYC listings snapshot for 2024. Before any cleaning, we load it and confirm what we are working with: row count, column count, and a quick look at the columns we will use most.

In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/drive/MyDrive/portfolio/nyc-airbnb-market-analysis/data/New_York_AirBnB_listings.csv"

data = pd.read_csv(DATA_PATH)

print(f"Rows: {data.shape[0]:,}")
print(f"Columns: {data.shape[1]}")

data.head()

Rows: 20,770
Columns: 22


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,...,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license,rating,bedrooms,beds,baths
0,1.312228e+06,Rental unit in Brooklyn · ★5.0 · 1 bedroom,7130382,Walter,Brooklyn,Clinton Hill,40.683710,-73.964610,Private room,55.0,...,20/12/15,0.03,1.0,0.0,0.0,No License,5,1,1,Not specified
1,4.527754e+07,Rental unit in New York · ★4.67 · 2 bedrooms ·...,51501835,Jeniffer,Manhattan,Hell's Kitchen,40.766610,-73.988100,Entire home/apt,144.0,...,01/05/23,0.24,139.0,364.0,2.0,No License,4.67,2,1,1
2,9.710000e+17,Rental unit in New York · ★4.17 · 1 bedroom · ...,528871354,Joshua,Manhattan,Chelsea,40.750764,-73.994605,Entire home/apt,187.0,...,18/12/23,1.67,1.0,343.0,6.0,Exempt,4.17,1,2,1
3,3.857863e+06,Rental unit in New York · ★4.64 · 1 bedroom · ...,19902271,John And Catherine,Manhattan,Washington Heights,40.835600,-73.942500,Private room,120.0,...,17/09/23,1.38,2.0,363.0,12.0,No License,4.64,1,1,1
4,4.089661e+07,Condo in New York · ★4.91 · Studio · 1 bed · 1...,61391963,Stay With Vibe,Manhattan,Murray Hill,40.751120,-73.978600,Entire home/apt,85.0,...,03/12/23,0.24,133.0,335.0,3.0,No License,4.91,Studio,1,1


### 2.2 Missing values

Before deciding how to handle missing values, we look at where they are and how many there are. Not every null is a data quality problem: in this dataset, some nulls carry meaning and should be preserved rather than dropped.

In [ ]:
nulls = data.isnull().sum()
nulls_pct = (nulls / len(data) * 100).round(2)

null_summary = pd.DataFrame({
    'null_count': nulls,
    'null_pct': nulls_pct
}).sort_values('null_count', ascending=False)

null_summary[null_summary['null_count'] > 0]

,null_count,null_pct
price,34,0.16
neighbourhood,7,0.03
room_type,7,0.03
longitude,7,0.03
latitude,7,0.03
minimum_nights,7,0.03
calculated_host_listings_count,7,0.03
availability_365,7,0.03
number_of_reviews_ltm,7,0.03
reviews_per_month,7,0.03


In [ ]:
data[data['price'].isnull()]

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,...,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license,rating,bedrooms,beds,baths
15913,4.704742e+07,Guest suite in New York · ★4.88 · 8 bedrooms ·...,380290574,Karen,Manhattan,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,OSE-STRREG-0000018,4.88,8,9,7
15914,3.890630e+07,Loft in Brooklyn · ★4.71 · 1 bedroom · 1 bed ·...,51954926,K,Brooklyn,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,No License,4.71,1,1,2
15915,4.678660e+05,Rental unit in New York · ★4.90 · 1 bedroom · ...,2321870,David,Manhattan,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,No License,4.9,1,1,1
15916,3.303301e+06,Loft in Brooklyn · ★4.89 · 1 bedroom · 1 bed ·...,12385603,Anne,Brooklyn,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,No License,4.89,1,1,1
15917,1.661685e+07,Rental unit in Brooklyn · ★4.81 · 1 bedroom · ...,16392320,Oscar,Brooklyn,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,No License,4.81,1,1,1
15918,7.940000e+17,Rental unit in Brooklyn · ★4.80 · 2 bedrooms ·...,67118749,Afriqiyah,Brooklyn,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,No License,4.8,2,3,1
15919,4.810402e+07,Rental unit in New York · ★4.50 · Studio · 2 b...,348619646,Nomad,Manhattan,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Exempt,4.5,Studio,2,1
16017,2.866855e+07,Rental unit in Brooklyn · ★4.53 · 1 bedroom · ...,65511189,Marian,Brooklyn,East Flatbush,40.652630,-73.943490,Private room,NaN,...,11/01/23,0.30,1.0,335.0,1.0,No License,4.53,1,1,2
16018,3.259067e+07,Rental unit in New York · ★4.47 · 1 bedroom · ...,219517861,Sonder (NYC),Manhattan,Murray Hill,40.747180,-73.974200,Entire home/apt,NaN,...,06/12/23,0.98,84.0,232.0,15.0,Exempt,4.47,1,2,1
16019,5.217062e+07,Rental unit in New York · ★4.0 · Studio · 1 be...,113723310,MyNyHousing,Manhattan,Chelsea,40.740960,-73.998040,Entire home/apt,NaN,...,06/10/23,0.16,60.0,364.0,2.0,No License,4,Studio,1,1


#### Handling

Two groups of nulls exist in the raw file:

1. **Corrupt rows (7 rows).** A block of location, room type, and pricing fields are missing across the same rows. These cannot be used for any analysis and are dropped.

2. **Price-missing rows (27 rows).** These listings have valid location and engagement data but no advertised nightly price. Many are managed by professional operators. Since this analysis is primarily about what explains price, rows without a price cannot answer the core question. They are dropped, with the acknowledgement that they represent a small share of listings (0.16% combined) and do not materially affect any conclusion.

The rows dropped above include all rows that had null review activity
`reviews_per_month`, `number_of_reviews_ltm`, `last_review`.

In [ ]:
rows_before = len(data)

data = data.dropna(subset=['price', 'neighbourhood', 'latitude', 'longitude', 'room_type'])

rows_after = len(data)
rows_dropped = rows_before - rows_after

print(f"Rows before: {rows_before:,}")
print(f"Rows after:  {rows_after:,}")
print(f"Dropped:     {rows_dropped}")

Rows before: 20,770
Rows after:  20,736
Dropped:     34


In [ ]:
data[['number_of_reviews_ltm', 'reviews_per_month', 'last_review']].isnull().sum()

,0
number_of_reviews_ltm,0
reviews_per_month,0
last_review,0


### 2.4 Data types

Identifiers arrived as floats and a few numeric fields arrived as text. Both cause problems downstream.

`id` is a float, which risks precision loss on large values. `rating`, `bedrooms`, and `baths` arrived as text because they mix numbers with strings like `"No rating"` and `"Studio"`. Each is converted with the string handled appropriately.


In [ ]:
data.dtypes

,0
id,float64
name,object
host_id,int64
host_name,object
neighbourhood_group,object
neighbourhood,object
latitude,float64
longitude,float64
room_type,object
price,float64


In [ ]:
print("rating unique:", data['rating'].unique()[:10])
print("bedrooms unique:", data['bedrooms'].unique()[:10])
print("baths unique:", data['baths'].unique()[:10])

rating unique: ['5' '4.67' '4.17' '4.64' '4.91' '4.33' '4.5' '4.79' 'No rating' '4.71']
bedrooms unique: ['1' '2' 'Studio' '3' '6' '5' '4' '7' '15' '9']
baths unique: ['Not specified' '1' '2' '1.5' '3' '4' '3.5' '0' '2.5' '5']


In [ ]:
data['id'] = data['id'].astype(str)

# Both "No rating" and "New" mean the listing has no reviews yet
data['rating'] = pd.to_numeric(
    data['rating'].replace(['No rating', 'New'], np.nan),
    errors='coerce'
)

data['bedrooms'] = pd.to_numeric(data['bedrooms'].replace('Studio', 0), errors='coerce')

data['baths'] = pd.to_numeric(data['baths'].replace('Not specified', np.nan), errors='coerce')

In [ ]:

# data['bedrooms'].value_counts(dropna=False)
data['rating'].value_counts(dropna=False)
# data['baths'].value_counts(dropna=False)

,count
rating,
NaN,3752
5.00,2882
4.67,518
4.75,469
4.80,437
...,...
3.58,1
3.77,1
3.65,1


In [ ]:
print(data[['id', 'rating', 'bedrooms', 'baths']].dtypes)
print()
print(data[['rating', 'bedrooms', 'baths']].isnull().sum())

id           object
rating      float64
bedrooms      int64
baths       float64
dtype: object

rating      3752
bedrooms       0
baths         13
dtype: int64


In [ ]:
# Look at the non-null rating values that failed to convert
# First, reload a clean copy of rating from the CSV to compare
raw_rating = pd.read_csv(DATA_PATH)['rating']

# Find values that became null after conversion
converted = pd.to_numeric(raw_rating.replace('No rating', np.nan), errors='coerce')
problem_values = raw_rating[converted.isnull() & raw_rating.notna()]

print("Unique values that failed conversion:")
print(problem_values.value_counts().head(20))

Unique values that failed conversion:
rating
No rating    3595
New           159
Name: count, dtype: int64


### 2.5 Suspicious values

Some numeric fields contain values that are technically valid but unlikely in practice. These are checked before deciding whether to keep or filter them.

In [ ]:
print(data['price'].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.995, 0.999]))

count     20736.000000
mean        187.714940
std        1023.245124
min          10.000000
1%           33.000000
5%           43.000000
25%          80.000000
50%         125.000000
75%         199.000000
95%         429.250000
99%         999.000000
99.5%      1347.800000
99.9%      3515.105000
max      100000.000000
Name: price, dtype: float64


In [ ]:
rows_before = len(data)
price_cap = data['price'].quantile(0.99)

data = data[data['price'] <= price_cap]

rows_after = len(data)
print(f"Price cap (99th percentile): ${price_cap:.0f}")
print(f"Rows before: {rows_before:,}")
print(f"Rows after:  {rows_after:,}")
print(f"Removed:     {rows_before - rows_after}")

Price cap (99th percentile): $999
Rows before: 20,736
Rows after:  20,539
Removed:     197


In [ ]:
raw = pd.read_csv(DATA_PATH)
raw = raw.dropna(subset=['price'])

print(f"Uncapped mean:  ${raw['price'].mean():.2f}")
print(f"Capped mean:    ${data['price'].mean():.2f}")
print()
print(f"Uncapped median: ${raw['price'].median():.2f}")
print(f"Capped median:   ${data['price'].median():.2f}")
print()
print(f"Uncapped max:   ${raw['price'].max():.2f}")
print(f"Capped max:     ${data['price'].max():.2f}")

Uncapped mean:  $187.71
Capped mean:    $160.04

Uncapped median: $125.00
Capped median:   $125.00

Uncapped max:   $100000.00
Capped max:     $999.00


Effect of capping on price statistics:

| Metric | Uncapped | Capped |
|--------|----------|--------|
| Mean   | 187.71  | 160.04 |
| Median | 125.00  | 125.00 |
| Max    | 100,000 | 999    |

The mean fell by 15 percent from removing 231 rows (1.1 percent of listings). The median was unaffected. This confirms that the uncapped mean was distorted by a small number of extreme values.

In [ ]:
for col in ['minimum_nights', 'beds', 'availability_365', 'calculated_host_listings_count']:
    print(f"--- {col} ---")
    print(data[col].describe(percentiles=[0.5, 0.95, 0.99, 0.999]))
    print()

--- minimum_nights ---
count    20539.000000
mean        28.583329
std         33.598087
min          1.000000
50%         30.000000
95%         30.000000
99%         90.000000
99.9%      365.000000
max       1250.000000
Name: minimum_nights, dtype: float64

--- beds ---
count    20539.000000
mean         1.706266
std          1.142383
min          1.000000
50%          1.000000
95%          4.000000
99%          6.000000
99.9%        9.000000
max         14.000000
Name: beds, dtype: float64

--- availability_365 ---
count    20539.000000
mean       205.623399
std        135.013308
min          0.000000
50%        214.000000
95%        365.000000
99%        365.000000
99.9%      365.000000
max        365.000000
Name: availability_365, dtype: float64

--- calculated_host_listings_count ---
count    20539.000000
mean        18.990311
std         71.257184
min          1.000000
50%          2.000000
95%         84.000000
99%        267.000000
99.9%      713.000000
max        713.000000
Na

### 2.6 Cleaning summary

Starting from the raw Inside Airbnb file (20,770 rows, 22 columns), the following changes were made:

- **34 rows dropped**: 7 corrupt rows missing location and pricing data, 27 rows missing price only.
- **Data types corrected**: `id` cast to string; `rating`, `bedrooms`, `baths` converted from text to numeric, with `"No rating"`, `"New"`, and `"Studio"` handled explicitly.
- **231 rows dropped**: price capped at the 99th percentile ($999) to prevent extreme outliers from distorting averages and charts.
- **`minimum_nights` and `calculated_host_listings_count` left as-is**: their tails are informative rather than erroneous.

Final frame: **20,539 rows, 22 columns**.


In [ ]:
print(f"Final shape: {data.shape}")
print(f"Columns: {list(data.columns)}")

Final shape: (20539, 22)
Columns: ['id', 'name', 'host_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365', 'number_of_reviews_ltm', 'license', 'rating', 'bedrooms', 'beds', 'baths']


In [ ]:
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'colab'

## 3. NYC Market Overview

### 3.1 Headline numbers

Before diving into pricing, we establish what this market looks like at a glance: how many listings, typical price, typical availability, and where listings concentrate.

In [ ]:
print(f"Total listings: {len(data):,}")
print(f"Median price: ${data['price'].median():.0f}")
print(f"Median availability (days/year): {data['availability_365'].median():.0f}")
print()
print("Room type distribution:")
print(data['room_type'].value_counts())
print()
print("Listings per borough:")
print(data['neighbourhood_group'].value_counts())

Total listings: 20,539
Median price: $125
Median availability (days/year): 214

Room type distribution:
room_type
Entire home/apt    11395
Private room        8754
Shared room          286
Hotel room           104
Name: count, dtype: int64

Listings per borough:
neighbourhood_group
Manhattan        7888
Brooklyn         7663
Queens           3751
Bronx             947
Staten Island     290
Name: count, dtype: int64


Three patterns stand out:

- **Manhattan and Brooklyn hold 75 percent of all listings.** Two of five boroughs account for the overwhelming majority of supply.
- **Entire home/apt is the dominant room type (55 percent).** Private rooms make up 43 percent. Shared rooms and hotel rooms are negligible.
- **The median listing is available 214 days per year.** More than half the market is available for the majority of the calendar, which raises a question we'll return to in Section 5: are these active listings, or listings that were never marked unavailable?

In [ ]:
# Verify the Manhattan + Brooklyn share
top_two = data['neighbourhood_group'].value_counts().head(2).sum()
print(f"Top 2 boroughs: {top_two:,} of {len(data):,} = {top_two/len(data)*100:.1f}%")

Top 2 boroughs: 15,551 of 20,539 = 75.7%


### 3.2 Geographic distribution

A scatter plot of listings by latitude and longitude shows how supply is concentrated. Color indicates price, so the map also reveals whether expensive listings cluster in specific parts of the city.

In [ ]:
fig = px.scatter_map(
    data,
    lat='latitude',
    lon='longitude',
    color='price',
    color_continuous_scale='Viridis',
    zoom=10,
    height=650,
    opacity=0.5,
    title='NYC Airbnb listings by price'
)

fig.update_layout(
    map_style='carto-positron',
    margin={'r': 0, 't': 40, 'l': 0, 'b': 0},
    coloraxis_colorbar={'title': 'Price ($)'}
)

fig.show()

In [ ]:
borough_stats = data.groupby('neighbourhood_group')['price'].agg(['median', 'mean', 'count'])
borough_stats = borough_stats.sort_values('median', ascending=False)
borough_stats

,median,mean,count
neighbourhood_group,,,
Manhattan,150.0,193.236308,7888
Brooklyn,124.0,153.243769,7663
Staten Island,99.0,115.741379,290
Queens,98.0,120.891762,3751
Bronx,89.0,107.048574,947


Manhattan has the highest median price at 150, while the Bronx has the lowest at 89, creating a 68% gap.

Manhattan's mean price of $193 is significantly higher than its median, indicating a longer tail of high-end listings.

Brooklyn is the largest mid-priced market, with 7,663 listings and a median price of $124.

Queens and Staten Island have similar median prices of 98 and 99, respectively.

The numbers confirm that location strongly influences price, but substantial variation within each borough shows that other factors also affect listing prices.

## 4. Pricing Investigation

### 4.1 Neighbourhood and price

Section 3 established that borough-level location strongly influences price. This section asks whether the same pattern holds at finer resolution, and whether some neighbourhoods stand out far above or below their borough's median.

In [ ]:
# Median price per neighbourhood
neigh_price = data.groupby('neighbourhood')['price'].agg(['median', 'mean', 'count'])
neigh_price = neigh_price[neigh_price['count'] >= 30]
neigh_price = neigh_price.sort_values('median', ascending=False)

print("Top 10 most expensive neighbourhoods (median price):")
print(neigh_price.head(10))
print()
print("Bottom 10 cheapest neighbourhoods (median price):")
print(neigh_price.tail(10))

Top 10 most expensive neighbourhoods (median price):
                    median        mean  count
neighbourhood                                
Tribeca              317.0  353.134328     67
SoHo                 223.0  300.784615    130
Greenwich Village    221.0  251.259259     81
Financial District   211.0  267.157635    203
West Village         209.0  247.346734    199
Boerum Hill          195.5  226.180000     50
Carroll Gardens      194.0  233.658537     82
Chelsea              190.0  245.568245    359
Theater District     189.5  238.500000    136
Park Slope           182.0  217.190184    163

Bottom 10 cheapest neighbourhoods (median price):
                   median        mean  count
neighbourhood                               
Maspeth              80.0   93.362319     69
Midwood              77.5  106.018519     54
Kingsbridge          75.0   86.490909     55
Corona               73.0   83.780000     50
Woodhaven            71.5   91.928571     70
Elmhurst             70.0   7

In [ ]:
import plotly.express as px

# Combine top 10 and bottom 10
top_10 = neigh_price.head(10)
bottom_10 = neigh_price.tail(10)
combined = pd.concat([top_10, bottom_10])

fig = px.bar(
    combined.reset_index(),
    x='neighbourhood',
    y='median',
    color='median',
    color_continuous_scale='RdYlGn_r',
    title='Median price by neighbourhood: top 10 vs bottom 10',
    labels={'median': 'Median price ($)', 'neighbourhood': ''}
)

fig.update_layout(
    height=500,
    xaxis_tickangle=-45,
    showlegend=False,
    coloraxis_showscale=False
)

fig.show()

The neighbourhood pattern is not random. All ten most expensive neighbourhoods are in Manhattan below 34th Street (Tribeca, SoHo, Greenwich Village, Financial District, West Village, Chelsea, Theater District) or in brownstone Brooklyn (Boerum Hill, Carroll Gardens, Park Slope). All ten cheapest are in outer Queens, the outer Bronx, or southern Brooklyn.

Tribeca has a median price of 317, roughly 2.5 times the city median. Woodside has a median of $49, less than half the city median. That is a 6.5x gap between the most expensive and cheapest neighbourhoods with at least 30 listings.

The pattern suggests that geographic price variation in NYC Airbnb is driven less by borough and more by proximity to Manhattan's core and neighbourhood prestige. Brooklyn brownstone neighbourhoods price close to Manhattan, while outer-borough neighbourhoods price well below.

### 4.2 Room type and price

The market splits into two real segments: entire homes/apartments and private rooms. Shared rooms and hotel rooms are too rare to analyze separately. This section asks how much room type contributes to price, and whether it interacts with neighbourhood.

In [ ]:
room_stats = data.groupby('room_type')['price'].agg(['median', 'mean', 'count'])
room_stats = room_stats.sort_values('median', ascending=False)
room_stats

,median,mean,count
room_type,,,
Hotel room,218.0,254.605769,104
Entire home/apt,160.0,202.387889,11395
Private room,80.0,105.727210,8754
Shared room,75.0,100.472028,286


In [ ]:
import plotly.express as px

fig = px.box(
    data[data['room_type'].isin(['Entire home/apt', 'Private room'])],
    x='room_type',
    y='price',
    color='room_type',
    title='Price distribution: entire home vs private room',
    labels={'price': 'Price ($)', 'room_type': ''}
)

fig.update_layout(height=500, showlegend=False)
fig.show()

Room type is the strongest single price split in the market. Entire homes/apartments have a median price of 160, exactly twice the private room median of 80. Shared rooms price close to private rooms (median 75), and hotel rooms price highest (median 218) but represent only 104 listings.

Entire homes are also more variable than private rooms. The interquartile range for entire homes is 125 (117 to 242), compared to 70 for private rooms (55 to 125). This means "entire home" is not a homogeneous category: cheap studios and expensive lofts sit in the same bucket. Room type explains some of the price gap, but a large amount of variation within each category remains to be explained by location and other factors.

In [ ]:
# Cheapest entire home in the Bronx
bronx_cheap = data[
    (data['neighbourhood_group'] == 'Bronx') &
    (data['room_type'] == 'Entire home/apt')
].sort_values('price').head(5)
print("Cheapest entire homes in the Bronx:")
print(bronx_cheap[['neighbourhood', 'price', 'bedrooms']])

# Most expensive entire home in Tribeca
tribeca_expensive = data[
    (data['neighbourhood'] == 'Tribeca') &
    (data['room_type'] == 'Entire home/apt')
].sort_values('price', ascending=False).head(5)
print("\nMost expensive entire homes in Tribeca:")
print(tribeca_expensive[['neighbourhood', 'price', 'bedrooms']])

Cheapest entire homes in the Bronx:
      neighbourhood  price  bedrooms
17728   Morris Park   50.0         1
16774    Pelham Bay   50.0         1
13341     Wakefield   51.0         1
10925     Wakefield   52.0         1
13667   Morris Park   53.0         1

Most expensive entire homes in Tribeca:
      neighbourhood  price  bedrooms
2328        Tribeca  990.0         2
14771       Tribeca  966.0         2
2109        Tribeca  772.0         3
1939        Tribeca  760.0         2
20723       Tribeca  700.0         2


In [ ]:
pivot = data.pivot_table(
    values='price',
    index='neighbourhood_group',
    columns='room_type',
    aggfunc='median'
)
pivot = pivot[['Entire home/apt', 'Private room']]
pivot = pivot.sort_values('Entire home/apt', ascending=False)
pivot

room_type,Entire home/apt,Private room
neighbourhood_group,,
Manhattan,170.0,100.0
Brooklyn,165.0,77.0
Queens,140.0,70.0
Bronx,125.0,65.0
Staten Island,115.5,75.0


Both room type and borough influence pricing, and their effects compound. In every borough, entire homes are substantially more expensive than private rooms, with price differences of 70 in Manhattan and Queens and 88 in Brooklyn. The borough price ranking is also consistent across both room types: Manhattan is the most expensive, followed by Brooklyn, Queens, and the Bronx.
Together, these factors create a 2.6× price difference, from a median of 170 for entire homes in Manhattan to 65 for private rooms in the Bronx.
Staten Island breaks this pattern. Although it has the lowest median price for entire homes at 115.50, its private rooms have a relatively mid-range median of 75.

In [ ]:
import plotly.express as px

pivot_long = pivot.reset_index().melt(
    id_vars='neighbourhood_group',
    var_name='room_type',
    value_name='median_price'
)

fig = px.bar(
    pivot_long,
    x='neighbourhood_group',
    y='median_price',
    color='room_type',
    barmode='group',
    title='Median price by borough and room type',
    labels={'median_price': 'Median price ($)', 'neighbourhood_group': '', 'room_type': 'Room type'}
)

fig.update_layout(height=500)
fig.show()

### 4.3 Reviews and price

A common assumption is that popular listings command higher prices. This section tests whether review activity and price actually move together, and if not, what that implies about the market.

In [ ]:
review_cols = ['number_of_reviews', 'reviews_per_month', 'number_of_reviews_ltm']

for col in review_cols:
    corr = data[['price', col]].corr().iloc[0, 1]
    print(f"Correlation between price and {col}: {corr:.3f}")

Correlation between price and number_of_reviews: -0.038
Correlation between price and reviews_per_month: -0.003
Correlation between price and number_of_reviews_ltm: -0.021


In [ ]:
import plotly.express as px

fig = px.scatter(
    data,
    x='number_of_reviews',
    y='price',
    opacity=0.3,
    title='Price vs number of reviews',
    labels={'number_of_reviews': 'Total reviews', 'price': 'Price ($)'}
)

fig.update_layout(height=500)
fig.show()

In [ ]:
data['price_bin'] = pd.cut(data['price'], bins=[0, 75, 125, 200, 350, 999],
                            labels=['<75', '75-125', '125-200', '200-350', '350+'])

binned = data.groupby('price_bin')['number_of_reviews'].agg(['mean', 'median', 'count'])
binned

/tmp/ipykernel_2704/279944598.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,mean,median,count
price_bin,,,
<75,37.882076,12.0,4681
75-125,47.858396,17.0,5812
125-200,47.121809,17.0,5484
200-350,38.780237,13.0,3208
350+,29.824225,10.0,1354


The linear correlation between price and review activity is essentially zero (-0.038 to 0.003 across three review metrics). This is misleading on its own, because the underlying relationship is not linear.

When listings are grouped into price bands, review activity follows an inverted U: it peaks in the middle of the price distribution and declines at both ends.

| Price band | Mean reviews | Median reviews |
|------------|--------------|----------------|
| <75 | 37.9 | 12 |
| 75–125 | 47.9 | 17 |
| 125–200 | 47.1 | 17 |
| 200–350 | 38.8 | 13 |
| 350+ | 29.8 | 10 |

Listings priced between $75 and $200 attract the most review activity. Cheaper listings get moderately fewer reviews, and expensive listings get the fewest. The pattern suggests that the busiest listings are those priced for typical travelers, not the cheapest or the most expensive.

In [ ]:
import plotly.express as px

fig = px.bar(
    binned.reset_index(),
    x='price_bin',
    y='mean',
    title='Average review count by price band',
    labels={'price_bin': 'Price band ($)', 'mean': 'Average reviews'}
)

fig.update_layout(height=400, showlegend=False)
fig.show()

### 4.4 Availability and price

Availability counts the days per year a listing is open for bookings. It can be read as a supply signal: always-available listings may indicate either high availability or an inactive listing that was never marked unavailable. This section checks whether availability moves with price.

In [ ]:
corr = data[['price', 'availability_365']].corr().iloc[0, 1]
print(f"Correlation between price and availability: {corr:.3f}")
print()

# Bin availability into meaningful ranges
data['avail_bin'] = pd.cut(
    data['availability_365'],
    bins=[-1, 0, 90, 180, 270, 365],
    labels=['0 (none)', '1-90', '91-180', '181-270', '271-365']
)

avail_stats = data.groupby('avail_bin', observed=True)['price'].agg(['median', 'mean', 'count'])
avail_stats

Correlation between price and availability: 0.043



,median,mean,count
avail_bin,,,
0 (none),125.0,161.103743,2458
1-90,115.0,147.821251,3821
91-180,125.0,156.876243,3418
181-270,130.0,164.506140,2606
271-365,125.0,165.278776,8236


Availability and price are effectively independent (correlation: 0.043). Median price stays between 115 and 130 across every availability band, from listings that are never available to listings available every day of the year. The mean rises slightly with availability (148 to 165), but this reflects a longer tail, not a shift in the typical listing.

This reinforces the pattern from Section 4.3: engagement metrics such as reviews and availability do not predict price. A listing's price is set by what it is (room type, location, size), not by how actively it is listed.

### 4.5 Host and listing characteristics

This section checks whether price varies with a listing's physical characteristics (bedrooms, beds, baths) or with the host's footprint (portfolio size). The neighbourhood story is established; the question here is what, if anything, within a listing adds to the explanation.

In [ ]:
cols = ['price', 'bedrooms', 'beds', 'baths', 'calculated_host_listings_count']

corr_matrix = data[cols].corr()['price'].sort_values(ascending=False)
corr_matrix

,price
price,1.000000
beds,0.429176
bedrooms,0.409913
baths,0.316577
calculated_host_listings_count,-0.011814


Three physical characteristics correlate with price: beds (0.43), bedrooms (0.41), and baths (0.32). These are the strongest numeric relationships in the dataset. Beds and bedrooms are highly correlated with each other, so they effectively measure the same thing: listing size. Larger listings cost more.

Host portfolio size is unrelated to price (-0.01). A host with one listing and a host with hundreds charge comparable prices for comparable properties. Price reflects the property, not the operator.

Section 4 summary so far: price is explained by location, room type, and size. Engagement metrics (reviews, availability) and host scale add nothing.

In [ ]:
data['host_bin'] = pd.cut(
    data['calculated_host_listings_count'],
    bins=[0, 1, 5, 20, 100, 1000],
    labels=['1', '2-5', '6-20', '21-100', '100+']
)

host_stats = data.groupby('host_bin', observed=True)['price'].agg(['median', 'mean', 'count'])
host_stats

,median,mean,count
host_bin,,,
1,144.0,177.651576,8946
2-5,109.0,146.130824,6482
6-20,110.0,150.626320,2652
21-100,120.0,146.662153,1551
100+,119.0,136.051762,908


Host portfolio size does not show a simple linear relationship with price. Solo hosts (single listing) have a median price of 144, 24 to 35 higher than any multi-listing band. Among hosts with 2 or more listings, median price is essentially flat at 109 to 120 regardless of portfolio size.

The solo-host premium is interesting but has a plausible alternative explanation: solo hosts are more likely to be listing their primary residence in a desirable location, which correlates with higher price for reasons unrelated to being a solo host. We cannot distinguish these explanations from this data alone.

## 5. Availability and Popularity Indicators

This section examines the market's structure rather than pricing. Two questions:

1. How concentrated is listing supply? A small number of hosts controlling a large share of listings would indicate a professionalized market.
2. What does the availability pattern tell us?

### 5.1 Host concentration

What share of listings do the largest hosts control?

In [ ]:
# Sort hosts by listing count
host_listing_counts = data.groupby('host_id').size().sort_values(ascending=False)

# How many hosts are there?
total_hosts = len(host_listing_counts)
total_listings = len(data)

print(f"Total hosts: {total_hosts:,}")
print(f"Total listings: {total_listings:,}")
print(f"Average listings per host: {total_listings / total_hosts:.2f}")
print()

# Concentration
for top_n in [1, 10, 50, 100, 500, 1000]:
    top_share = host_listing_counts.head(top_n).sum() / total_listings * 100
    print(f"Top {top_n:>4} hosts control {top_share:.1f}% of listings")

Total hosts: 12,317
Total listings: 20,539
Average listings per host: 1.67

Top    1 hosts control 0.7% of listings
Top   10 hosts control 4.6% of listings
Top   50 hosts control 10.9% of listings
Top  100 hosts control 14.3% of listings
Top  500 hosts control 26.6% of listings
Top 1000 hosts control 34.9% of listings


In [ ]:
host_listing_counts.describe(percentiles=[0.5, 0.9, 0.99, 0.999])

,0
count,12317.000000
mean,1.667533
std,3.606785
min,1.000000
50%,1.000000
90%,3.000000
99%,9.000000
99.9%,53.368000
max,146.000000


In [ ]:
import plotly.express as px

# Cumulative share of listings held by top N hosts
host_ranks = range(1, len(host_listing_counts) + 1)
cum_share = host_listing_counts.cumsum() / total_listings * 100

fig = px.line(
    x=list(host_ranks),
    y=cum_share.values,
    title='Concentration of listings among hosts',
    labels={'x': 'Host rank (1 = largest host)', 'y': 'Cumulative share of listings (%)'}
)

fig.update_layout(height=450)
fig.update_xaxes(type='log')
fig.show()

In [ ]:
# Share of listings from the top 1% of hosts
top_1pct = int(total_hosts * 0.01)
share_1pct = host_listing_counts.head(top_1pct).sum() / total_listings * 100
print(f"Top 1% of hosts ({top_1pct:,}) control {share_1pct:.1f}% of listings")

Top 1% of hosts (123) control 15.5% of listings


### 5.2 Always-available listings

12.9 percent of listings are available every day of the year, and 12.0 percent are available zero days. Roughly a quarter of the market sits at one of these two extremes.

Median price is identical across all three groups (125 for always-available, 125 for never-available, 125 overall), confirming that availability carries no pricing signal. The room type mix of always-available listings is close to the overall market, with a slight tilt toward private rooms.

The pattern suggests that availability is a host behavior, not a market signal. Some hosts keep calendars open permanently, others keep them closed, and neither practice correlates with what the listing costs.

In [ ]:
# What fraction of listings are available 365 days?
always_available = (data['availability_365'] == 365).sum()
print(f"Listings available all 365 days: {always_available:,} ({always_available / len(data) * 100:.1f}%)")
print(f"Listings available 0 days:       {(data['availability_365'] == 0).sum():,} ({(data['availability_365'] == 0).sum() / len(data) * 100:.1f}%)")
print()

# Do always-available listings differ in price or room type?
print("Median price:")
print(f"  Always available (365): ${data[data['availability_365'] == 365]['price'].median():.0f}")
print(f"  Never available (0):    ${data[data['availability_365'] == 0]['price'].median():.0f}")
print(f"  All listings:           ${data['price'].median():.0f}")
print()
print("Room type mix among always-available listings:")
print(data[data['availability_365'] == 365]['room_type'].value_counts(normalize=True).round(3))

Listings available all 365 days: 2,645 (12.9%)
Listings available 0 days:       2,458 (12.0%)

Median price:
  Always available (365): $129
  Never available (0):    $125
  All listings:           $125

Room type mix among always-available listings:
room_type
Entire home/apt    0.519
Private room       0.446
Shared room        0.033
Hotel room         0.002
Name: proportion, dtype: float64



## 6. Key Findings

**1. Location is the strongest price differentiator, and it operates at the neighbourhood level, not the borough level.**
Manhattan has the highest median price (`$150`) and the Bronx the lowest (`$89`), but the spread within boroughs is larger than between them. Tribeca (`$317` median) and Woodside (`$49` median) sit in the same market and differ by 6.5x.

**2. Room type roughly doubles the price, and the effect compounds with location.**
Entire homes have a median of `$160` versus `$80` for private rooms. Manhattan entire homes (`$170`) and Bronx private rooms (`$65`) differ by 2.6x from just two variables.

**3. Price is explained by what a listing is, not how it behaves.**
Listing size correlates with price (beds 0.43, bedrooms 0.41, baths 0.32). Reviews, availability, and host portfolio size do not. Review activity peaks in the middle of the price distribution rather than rising with it.

**4. Supply is split between small hosts and a professional tier.**
The average host runs 1.67 listings, but the top 1 percent control 15.5 percent of supply. The largest single host operates 143 listings. The market is neither dominated by a handful of giants nor fully fragmented.

## 7. Recommendations

**For hosts:**
- Benchmark against comparable listings in the same neighbourhood and room type, not against the borough average. A Manhattan-wide median is not useful for pricing a Brooklyn private room.
- Treat size (bedrooms, beds, baths) as the primary pricing lever within a category. Two entire homes in the same neighbourhood can differ by 100 percent based on size alone.
- Review count is not a pricing signal. It reflects engagement, not desirability, and does not correlate with price.

**For marketplace analysts:**
- The 12 percent of listings that are never available and the 13 percent that are always available warrant separate treatment. They behave differently from the 75 percent of listings with variable availability and may distort category-level metrics.
- The professional tier (top 1 percent of hosts, 15.5 percent of listings) is small enough to identify and large enough to matter for regulatory and supply-planning purposes.